# SatQuery AI — Qwen2.5-VL-3B Local Fine-Tuning
**VQA task only | RTX 3050 8GB | Local dataset**

| Setting | Value |
|---------|-------|
| Model | Qwen2.5-VL-3B-Instruct (4-bit) |
| Task | VQA only |
| LoRA rank | r=8 (low VRAM) |
| Vision encoder | Frozen (language-only LoRA) |
| Batch size | 1 × grad_accum 16 = 16 effective |
| Max seq length | 512 tokens |
| Dataset | Local VRSBench_train.json (VQA split) |

**Run cells top to bottom. Training takes ~6-8 hours on RTX 3050.**

## Cell 1 — Install Dependencies
Run once. Restart kernel after installation.

In [2]:
import subprocess, sys

# CUDA PyTorch is already installed from terminal.
# Just install the ML training packages.

for pkg in [
    "transformers==4.56.2",
    "trl==0.22.2",
    "peft",
    "accelerate",
    "bitsandbytes",
    "qwen-vl-utils",
    "Pillow",
    "tqdm",
    "sentencepiece",
    "protobuf",
]:
    print(f"Installing {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

# Install unsloth without deps (avoids xformers Windows long-path issue)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "unsloth_zoo", "unsloth", "--no-deps", "-q"
], check=False)

print("\nAll done! >>> Restart Kernel now, then run Cell 2")


Installing transformers==4.56.2...
Installing trl==0.22.2...
Installing peft...
Installing accelerate...
Installing bitsandbytes...
Installing qwen-vl-utils...
Installing Pillow...
Installing tqdm...
Installing sentencepiece...
Installing protobuf...

All done! >>> Restart Kernel now, then run Cell 2


## Cell 2 — GPU Check

In [1]:
import torch

assert torch.cuda.is_available(), "No GPU found! Make sure CUDA drivers are installed."

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

if vram_gb < 7:
    print("WARNING: Less than 7 GB VRAM detected — may OOM during training.")
else:
    print("VRAM OK for VQA-only fine-tuning with frozen vision encoder.")

GPU  : NVIDIA GeForce RTX 3050
VRAM : 8.0 GB
CUDA : 12.4
PyTorch: 2.6.0+cu124
VRAM OK for VQA-only fine-tuning with frozen vision encoder.


In [3]:
import subprocess, sys

# Triton doesn't work on Windows Python 3.13 — uninstall it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "triton", "-y"])
print("Triton removed. Now re-run Cell 3.")


Triton removed. Now re-run Cell 3.


## Cell 3 — Load Model + LoRA

In [5]:
from unsloth import FastVisionModel
import torch

print("Loading Qwen2.5-VL-3B-Instruct in 4-bit...")
print("(First run downloads ~3 GB — needs internet once)")

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",  # saves ~30% VRAM
)

# Add LoRA — LANGUAGE LAYERS ONLY (vision encoder frozen to save VRAM)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # FROZEN — saves ~3 GB VRAM
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,                               # Low rank for 8 GB VRAM
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
vram = torch.cuda.memory_allocated() / 1024**3

print(f"\nModel loaded!")
print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"  VRAM used: {vram:.1f} GB")

C:\Users\IPS\AppData\Local\Programs\Python\Python313\Lib\site-packages\unsloth\__init__.py:1543: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Exception: cannot import name 'AttrsDescriptor' from 'triton.compiler.compiler' (C:\Users\IPS\AppData\Local\Programs\Python\Python313\Lib\site-packages\triton\compiler\compiler.py)

## Cell 4 — Load Local VQA Dataset

In [ ]:
import json
import re
import os
from datasets import Dataset
from PIL import Image
from tqdm import tqdm

# ── Paths (your local dataset) ──
DATASET_DIR    = r"C:\Users\IPS\Desktop\dataset"
TRAIN_JSON     = os.path.join(DATASET_DIR, "VRSBench_train.json")
TRAIN_IMG_DIR  = os.path.join(DATASET_DIR, "Images_train", "Images_train")

# Verify
assert os.path.exists(TRAIN_JSON),    f"Not found: {TRAIN_JSON}"
assert os.path.isdir(TRAIN_IMG_DIR),  f"Not found: {TRAIN_IMG_DIR}"
print(f"Dataset found: {DATASET_DIR}")

# ── System prompt ──
RS_SYSTEM_PROMPT = (
    "You are SatQuery AI, a remote sensing image analysis expert specializing in "
    "satellite and aerial imagery interpretation. You can identify land cover types, "
    "objects, spatial relationships, and scene characteristics in remote sensing images. "
    "Provide accurate, concise answers based on visual evidence in the image."
)

def clean_text(text):
    cleaned = text.replace("<image>", "").strip()
    cleaned = re.sub(r"\[(vqa|caption|refer)\]\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"</?p>", "", cleaned)
    return cleaned.strip()

# ── Load JSON ──
print("\nLoading VRSBench_train.json...")
with open(TRAIN_JSON, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"  Total entries: {len(raw_data):,}")

# ── Filter VQA only + convert ──
print("\nConverting VQA samples...")
converted = []
skipped = missing = 0

for sample in tqdm(raw_data, desc="Processing"):
    convs = sample.get("conversations", [])
    image_name = sample.get("image", "")

    if not convs or len(convs) < 2 or not image_name:
        skipped += 1
        continue

    # VQA only
    tag = convs[0].get("value", "").lower()
    if "[vqa]" not in tag:
        continue

    img_path = os.path.join(TRAIN_IMG_DIR, image_name)
    if not os.path.exists(img_path):
        missing += 1
        continue

    human_text = clean_text(convs[0]["value"])
    gpt_text   = convs[1]["value"].strip()

    # NOTE: We store the image PATH — images are loaded lazily during training
    # This avoids loading all 85K images into RAM at once
    converted.append({
        "image_path": img_path,
        "question":   human_text,
        "answer":     gpt_text,
    })

print(f"\nDataset ready!")
print(f"  VQA samples: {len(converted):,}")
print(f"  Skipped:     {skipped:,}")
print(f"  Img missing: {missing:,}")
print(f"\nSample entry:")
print(f"  Q: {converted[0]['question'][:80]}")
print(f"  A: {converted[0]['answer']}")

## Cell 5 — Build Training Dataset (with lazy image loading)

In [ ]:
from torch.utils.data import Dataset as TorchDataset
from PIL import Image

class VQADataset(TorchDataset):
    """
    Lazy-loading dataset: images are read from disk per sample.
    This keeps RAM usage low (never loads all images at once).
    """
    def __init__(self, samples, tokenizer, system_prompt, max_samples=None):
        self.samples = samples[:max_samples] if max_samples else samples
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        # Load image lazily
        image = Image.open(item["image_path"]).convert("RGB")

        # Build messages in Qwen2.5-VL format
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": self.system_prompt}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text":  item["question"]},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": item["answer"]}],
            },
        ]

        return {"messages": messages}

# Build dataset
train_dataset = VQADataset(converted, tokenizer, RS_SYSTEM_PROMPT)

print(f"Training dataset: {len(train_dataset):,} samples")
print(f"Test load sample 0...")
sample = train_dataset[0]
print(f"  Image size: {sample['messages'][1]['content'][0]['image'].size}")
print(f"  Question: {sample['messages'][1]['content'][1]['text'][:60]}...")
print(f"  Answer: {sample['messages'][2]['content'][0]['text']}")
print("Lazy loading OK!")

## Cell 6 — Train (~6-8 hours on RTX 3050)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported, UnslothVisionDataCollator
import torch

use_bf16 = is_bfloat16_supported()
print(f"Precision : {'bf16' if use_bf16 else 'fp16'}")
print(f"Samples   : {len(train_dataset):,}")
print(f"Eff batch : 1 x 16 grad_accum = 16")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    args=SFTConfig(
        # ── Batch (8 GB VRAM safe) ──
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,   # Effective batch = 16

        # ── Schedule ──
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,
        max_grad_norm=1.0,

        # ── Precision ──
        fp16=not use_bf16,
        bf16=use_bf16,

        # ── Sequence length (shorter = less VRAM) ──
        max_seq_length=512,

        # ── Optimizer (8-bit saves VRAM) ──
        optim="adamw_8bit",

        # ── Logging & saving ──
        logging_steps=50,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=3,
        output_dir="satquery_checkpoints",

        # ── Misc ──
        seed=3407,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

print("\nStarting training...")
print("Tip: VRAM usage will peak in the first few steps.")
print("If you see CUDA OOM, run the OOM fix cell below before restarting.\n")

stats = trainer.train()

print(f"\nTraining complete!")
print(f"  Time  : {stats.metrics['train_runtime']/3600:.1f} hours")
print(f"  Loss  : {stats.metrics['train_loss']:.4f}")
print(f"  VRAM  : {torch.cuda.max_memory_allocated()/1024**3:.1f} GB peak")

## Cell 6-OOM — If you get CUDA Out of Memory
Run this cell INSTEAD of Cell 6 if you get an OOM error.

In [ ]:
# Run this if Cell 6 crashes with CUDA OOM
# Reduces sequence length further and uses gradient checkpointing

import torch, gc
torch.cuda.empty_cache()
gc.collect()
print(f"VRAM freed. Available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3:.1f} GB")

from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported, UnslothVisionDataCollator

use_bf16 = is_bfloat16_supported()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not use_bf16,
        bf16=use_bf16,
        max_seq_length=256,             # Reduced from 512
        optim="adamw_8bit",
        logging_steps=50,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=3,
        output_dir="satquery_checkpoints",
        seed=3407,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

print("Retrying with max_seq_length=256...")
stats = trainer.train()
print(f"Done! Loss: {stats.metrics['train_loss']:.4f}")

## Cell 7 — Quick Inference Test

In [ ]:
from unsloth import FastVisionModel
from qwen_vl_utils import process_vision_info
from PIL import Image
import torch

FastVisionModel.for_inference(model)

def predict(image_path, question):
    image = Image.open(image_path).convert("RGB")
    messages = [
        {"role": "system", "content": RS_SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text":  question},
        ]},
    ]
    text_prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = tokenizer(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, temperature=0.1, do_sample=False)
    return tokenizer.batch_decode(
        out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0].strip()

# Test on 3 training samples
print("=" * 60)
print("  Inference Test (3 samples)")
print("=" * 60)

for i in [0, 100, 500]:
    s = converted[i]
    pred = predict(s["image_path"], s["question"])
    match = pred.lower().strip() == s["answer"].lower().strip()
    print(f"\nSample {i}:")
    print(f"  Q        : {s['question'][:70]}")
    print(f"  Expected : {s['answer']}")
    print(f"  Got      : {pred}")
    print(f"  Match    : {'YES' if match else 'NO'}")

## Cell 8 — Save Adapter Locally

In [ ]:
import os

ADAPTER_NAME = "satquery-ai-vqa-lora"
SAVE_DIR = os.path.join(r"C:\Users\IPS\Desktop\Satellite", ADAPTER_NAME)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(SAVE_DIR, f))
    for f in os.listdir(SAVE_DIR)
    if os.path.isfile(os.path.join(SAVE_DIR, f))
) / 1024**2

print(f"Adapter saved to: {SAVE_DIR}")
print(f"Size: {size_mb:.1f} MB")
print(f"Files: {os.listdir(SAVE_DIR)}")

## Cell 9 — Push to HuggingFace Hub (optional — needs internet)

In [ ]:
# Only run this when you have internet.
# Get token from: https://huggingface.co/settings/tokens

from huggingface_hub import login

login()  # Paste your HF write token when prompted

HF_REPO = "shivamw/satquery-ai-vqa-lora"

print(f"Pushing to https://huggingface.co/{HF_REPO} ...")
model.push_to_hub(HF_REPO, tokenizer=tokenizer)
print(f"Done! FastAPI backend will auto-download from this repo.")